In [17]:
import pandas as pd

columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
hospitals = ["cleveland", "hungarian", "switzerland", "va"]

frames = []
for h in hospitals:
    part = pd.read_csv(base_url + f"processed.{h}.data", header=None,
                       names=columns, na_values="?")
    part["source"] = h
    frames.append(part)

df = pd.concat(frames, ignore_index=True)

print(df.shape)
print(df["source"].value_counts())
print(df.isna().sum())

(920, 15)
source
cleveland      303
hungarian      294
va             200
switzerland    123
Name: count, dtype: int64
age           0
sex           0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalach      55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
source        0
dtype: int64


In [18]:
import numpy as np 
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
df[["chol", "trestbps"]] = df[["chol", "trestbps"]].replace(0, np.nan)

print(df["target"].value_counts())
print((df.drop(columns=["source", "target"]).isna()
         .groupby(df["source"]).mean() * 100).round(0))

target
1    509
0    411
Name: count, dtype: int64
             age  sex   cp  trestbps   chol   fbs  restecg  thalach  exang  \
source                                                                       
cleveland    0.0  0.0  0.0       0.0    0.0   0.0      0.0      0.0    0.0   
hungarian    0.0  0.0  0.0       0.0    8.0   3.0      0.0      0.0    0.0   
switzerland  0.0  0.0  0.0       2.0  100.0  61.0      1.0      1.0    1.0   
va           0.0  0.0  0.0      28.0   28.0   4.0      0.0     26.0   26.0   

             oldpeak  slope    ca  thal  
source                                   
cleveland        0.0    0.0   1.0   1.0  
hungarian        0.0   65.0  99.0  90.0  
switzerland      5.0   14.0  96.0  42.0  
va              28.0   51.0  99.0  83.0  


In [23]:
from sklearn.model_selection import train_test_split

SEED = 8  

X = df.drop(columns=["target", "source"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

test_df = X_test.copy()
test_df["target"] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print(X_train.shape, X_test.shape)


(736, 13) (184, 13)


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

def make_pipeline(clf, scale=True):
    steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("clf", clf))
    return Pipeline(steps)

candidates = {
    "Logistic Regression": (make_pipeline(LogisticRegression(max_iter=2000)),
                            {"clf__C": [0.001, 0.003, 0.01, 0.03, 0.1, 1]}),
    "Decision Tree": (make_pipeline(DecisionTreeClassifier(random_state=SEED), scale=False),
                      {"clf__max_depth": [3, 4, 5, 6, 8], "clf__min_samples_leaf": [1, 5, 10]}),
    "KNN": (make_pipeline(KNeighborsClassifier()),
            {"clf__n_neighbors": [15, 21, 25, 31, 41], "clf__weights": ["uniform", "distance"]}),
    "Naive Bayes": (make_pipeline(GaussianNB()),
                    {"clf__var_smoothing": [1e-9, 1e-7, 1e-5]}),
    "Support Vector Machine": (make_pipeline(SVC(probability=True, random_state=SEED)),
                               {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.1]}),
}

best_models = {}
for name, (pipe, grid) in candidates.items():
    search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"{name}: best params {search.best_params_}, CV AUC {search.best_score_:.4f}")

Logistic Regression: best params {'clf__C': 0.03}, CV AUC 0.8944
Decision Tree: best params {'clf__max_depth': 4, 'clf__min_samples_leaf': 10}, CV AUC 0.8354
KNN: best params {'clf__n_neighbors': 41, 'clf__weights': 'distance'}, CV AUC 0.8911
Naive Bayes: best params {'clf__var_smoothing': 1e-05}, CV AUC 0.8608
Support Vector Machine: best params {'clf__C': 1, 'clf__gamma': 'scale'}, CV AUC 0.8955


In [30]:
import os, joblib
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

os.makedirs("model", exist_ok=True)
rows = []
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    joblib.dump(model, f"model/{name.lower().replace(' ', '_')}.joblib")

results = pd.DataFrame(rows).set_index("ML Model Name").round(4)
results

,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.8207,0.8997,0.8485,0.8235,0.8358,0.6387
Decision Tree,0.7663,0.8569,0.8391,0.7157,0.7725,0.5425
KNN,0.8424,0.8998,0.8614,0.8529,0.8571,0.6814
Naive Bayes,0.6196,0.8570,0.8478,0.3824,0.5270,0.3409
Support Vector Machine,0.8315,0.9030,0.8447,0.8529,0.8488,0.6586
